Environment note
----------------
All training and dataset generation for this project ran under WSL2
(Ubuntu 22.04) with GPU acceleration. Paths in those notebooks point
at /home/admins/ and reflect that environment.

Phase 4 onward runs on Windows, CPU only - inference and demo work,
no training. Paths here point at D:\lip_codebase_clean\.

The two environments were verified equivalent: the same model on the
same test set reproduces 0.7150 (full test) and 0.7556 (real
recordings only) to four decimal places on both.

One difference: albumentations does not install on Windows Python 3.11
(its stringzilla dependency needs a C++ compiler), so dataset
generation is WSL-only. The datasets are already built.

See docs/environment/ for exact package versions on both platforms.

In [75]:
import os
import sys
import time
import cv2
import numpy as np
import mediapipe as mp
from PIL import Image

CONFIG_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if CONFIG_DIR not in sys.path:
    sys.path.insert(0, CONFIG_DIR)

from config import ROOT, RUNTIME, FRAMES, CROPPED, GRIDS, SOURCE_VIDEOS
from pipeline import (clear_runtime, normalise_extracted_frames,
                      crop_extracted_frames, measure_mouth_boxes,
                      build_grid, load_lip_model, predict_word,
                      load_t5_model, generate_sentence)

**demo code**

In [76]:
# Open and configure the camera before starting the live capture cell
if "cap" in globals() and globals()["cap"] is not None:
    globals()["cap"].release()
    
initialization_started = time.perf_counter()

cap = cv2.VideoCapture(0, cv2.CAP_MSMF)
if not cap.isOpened():
    raise RuntimeError("MSMF could not open camera index 0")

cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)

ret, initialization_frame = cap.read()
if not ret:
    cap.release()
    raise RuntimeError("Camera opened but initialization frame could not be read")

print(f"camera initialization: {time.perf_counter() - initialization_started:.2f}s")
print(
    f"resolution: "
    f"{initialization_frame.shape[1]}x{initialization_frame.shape[0]}"
)
print(f"reported fps: {cap.get(cv2.CAP_PROP_FPS):.1f}")
print("camera ready")

camera initialization: 22.51s
resolution: 1920x1080
reported fps: 30.0
camera ready


In [88]:
# Run only when the complete camera session is finished
if "cap" in globals() and cap is not None and cap.isOpened():
    cap.release()

cv2.destroyAllWindows()
print("camera released")

camera released


*frame recording*

In [78]:
cell_started = time.perf_counter()

clear_runtime()

# Initialize MediaPipe Hands and Drawing modules
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

# Reuse the MSMF camera opened by the initialization cell
if "cap" not in globals() or cap is None or not cap.isOpened():
    raise RuntimeError("Run the camera initialization cell first")

# Variables to handle recording
recording = False
finished = False          # set once one full open -> fist cycle completes
captured_frames = []
t_start = None
elapsed = 0.0

def is_open_hand(hand_landmarks):
    """ Check if all fingers are extended (open hand) """
    for finger_tip, finger_pip in [
        (mp_hands.HandLandmark.INDEX_FINGER_TIP, mp_hands.HandLandmark.INDEX_FINGER_PIP),
        (mp_hands.HandLandmark.MIDDLE_FINGER_TIP, mp_hands.HandLandmark.MIDDLE_FINGER_PIP),
        (mp_hands.HandLandmark.RING_FINGER_TIP, mp_hands.HandLandmark.RING_FINGER_PIP),
        (mp_hands.HandLandmark.PINKY_TIP, mp_hands.HandLandmark.PINKY_PIP)
    ]:
        if hand_landmarks.landmark[finger_tip].y > hand_landmarks.landmark[finger_pip].y:
            return False
    return True

def is_closed_fist(hand_landmarks):
    """ Check if all fingers are folded (closed fist) """
    for finger_tip, finger_pip in [
        (mp_hands.HandLandmark.INDEX_FINGER_TIP, mp_hands.HandLandmark.INDEX_FINGER_PIP),
        (mp_hands.HandLandmark.MIDDLE_FINGER_TIP, mp_hands.HandLandmark.MIDDLE_FINGER_PIP),
        (mp_hands.HandLandmark.RING_FINGER_TIP, mp_hands.HandLandmark.RING_FINGER_PIP),
        (mp_hands.HandLandmark.PINKY_TIP, mp_hands.HandLandmark.PINKY_PIP)
    ]:
        if hand_landmarks.landmark[finger_tip].y < hand_landmarks.landmark[finger_pip].y:
            return False
    return True

preview_reported = False
mediapipe_started = time.perf_counter()

with mp_hands.Hands(min_detection_confidence=0.8, min_tracking_confidence=0.5) as hands:
    print(f"MediaPipe initialization: {time.perf_counter() - mediapipe_started:.2f}s")
    while cap.isOpened() and not finished:
        if not preview_reported:
            first_loop_started = time.perf_counter()
        ret, frame = cap.read()
        if not ret:
            break

        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image = cv2.flip(image, 1)
        image.flags.writeable = False
        results = hands.process(image)
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        if results.multi_hand_landmarks:
            for num, hand_landmarks in enumerate(results.multi_hand_landmarks):
                mp_drawing.draw_landmarks(
                    image, hand_landmarks, mp_hands.HAND_CONNECTIONS,
                    mp_drawing.DrawingSpec(color=(121, 22, 76), thickness=2, circle_radius=4),
                    mp_drawing.DrawingSpec(color=(121, 44, 250), thickness=2, circle_radius=2)
                )

                if is_open_hand(hand_landmarks):
                    cv2.putText(image, "Open Hand Detected", (10, 50),
                                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                    if not recording:
                        recording = True
                        captured_frames = []
                        t_start = time.time()
                        print("Recording started - speak now")

                elif is_closed_fist(hand_landmarks):
                    cv2.putText(image, "Closed Fist Detected", (10, 50),
                                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
                    if recording:
                        recording = False
                        finished = True
                        elapsed = time.time() - t_start
                        print(f"Recording stopped - {len(captured_frames)} frames "
                              f"in {elapsed:.2f}s ({len(captured_frames)/elapsed:.1f} fps)")

        if recording:
            captured_frames.append(frame.copy())
            elapsed = time.time() - t_start
            if len(captured_frames) >= 300:
                recording = False
                finished = True
                print(f"Recording stopped - frame limit reached at {elapsed:.2f}s")

        # timer and frame count overlay
        if recording:
            cv2.putText(image, f"REC  {elapsed:5.2f}s   {len(captured_frames)} frames",
                        (10, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)
            cv2.circle(image, (image.shape[1] - 40, 40), 12, (0, 0, 255), -1)
        elif not finished:
            cv2.putText(image, "Open hand to start", (10, 100),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (200, 200, 200), 2)

        cv2.imshow("Hand Gesture Recognition", image)
        key = cv2.waitKey(1) & 0xFF

        if not preview_reported:
            first_loop_elapsed = time.perf_counter() - first_loop_started
            preview_elapsed = time.perf_counter() - cell_started

            print(f"first processing loop: {first_loop_elapsed:.2f}s")
            print(f"time to first preview: {preview_elapsed:.2f}s")

            preview_reported = True

        if key == ord("q"):
            if recording:
                captured_frames = []
                recording = False
                print("recording cancelled")
            break

cv2.destroyAllWindows()

# save frames to disk and release memory
if captured_frames:
    for i, f in enumerate(captured_frames, start=1):
        cv2.imwrite(os.path.join(FRAMES, f"{i:02d}.png"), f)
    print(f"wrote {len(captured_frames)} frames to {FRAMES}")
    print(f"duration {elapsed:.2f}s")
    captured_frames = []
else:
    print("no frames captured")

removed 3 items from runtime/
MediaPipe initialization: 0.01s
first processing loop: 0.08s
time to first preview: 0.09s
Recording started - speak now
Recording stopped - 53 frames in 2.85s (18.6 fps)
wrote 53 frames to D:\lipreading_workbench\runtime\extracted_frames
duration 2.85s


*frame processing*

In [79]:
frame_stats = normalise_extracted_frames()

source frames : 53
padded        : 7
trimmed start : 0
trimmed end   : 0
written       : 60 to D:\lipreading_workbench\runtime\extracted_frames


In [80]:
crop_stats = crop_extracted_frames()

frames in     : 60
detected      : 60
failed frames : none
filled from   : none
written       : 60 to D:\lipreading_workbench\runtime\cropped_frames


In [81]:
box_w, box_h = measure_mouth_boxes()

frames measured : 60
source frame    : 1920x1080
mouth width  min/mean/max : 98 / 102.2 / 107
mouth height min/mean/max : 35 / 48.0 / 72
crop target     : 112x80


In [82]:
grid_path = build_grid()

cell size    : 112x80
grid         : 672x800 (10 rows x 6 cols)
resized      : 224x224
written      : D:\lipreading_workbench\runtime\grids


In [83]:
load_lip_model()

model  : model_rebuilt_data.h5
params : 51,434,058
input  : (None, 224, 224, 3)
labels : ['bat', 'cup', 'drop', 'eat', 'fish', 'hot', 'jump', 'milk', 'pen', 'red']


In [84]:
predicted_word, probabilities = predict_word()

predicted : fish  (0.8539)
top 3 :
  fish   0.8539
  eat    0.0601
  milk   0.0464


In [85]:
load_t5_model()

tokenizer : T5Tokenizer
model     : T5ForConditionalGeneration
loaded    : D:\lipreading_workbench\models\t5_fine_tuned_local


In [86]:
sentence = generate_sentence(predicted_word)

prompt    : Generate a sentence for fish:
sentence  : Can you add a little more to the fish for the day?


In [87]:
# sentence = generate_sentence("hot")